In [ ]:
import sys
sys.path.append("../preprocess")
import glob
from qbiolip import load_db
import re
import torch
df = load_db("../../databases/QBioLiP/qbiolip.csv", "../../databases/ec_uniprot_ids.txt")
df = df.loc[df.target]

result_path = "/home/daniil/Desktop/Ph.D/DynamicBind/results/*/*"
result_folders = glob.glob(result_path)
result_folders.sort()

targets = []
preds = []
names = []
def xyz_from_sdf(sdf_p):
    with open(sdf_p) as sdf_f:
        lines = [l.split() for l in sdf_f]
    lines = [l for l in lines if len(l) == 16] #only xyz coordinates lines
    xyz = torch.tensor([
        [
            float(l[0]), #x
            float(l[1]), #y
            float(l[2]), #z
        ]
    for l in lines])
    return xyz 

for folder in result_folders:
    if "affinity_prediction.csv" in folder:
        continue
    cases = glob.glob(f"{folder}/*")
    target = xyz_from_sdf(f"{folder}/ref_ligandFile.sdf")
    # target_file = [c for c in cases if re.search(r'')]
    protein_name = [c.split("/")[-1].replace(".pdb", "") for c in cases if re.search(r'^[0-9a-z]{4}_[0-9A-Z_]*.pdb', c.split("/")[-1]) is not None][0]
    ligand_name = [c.split("/")[-1].replace(".sdf", "") for c in cases if re.search(r'^[0-9a-z]{4}_[0-9A-Z_]*.sdf', c.split("/")[-1]) is not None][0]
    name = ligand_name if ligand_name != protein_name else f"{ligand_name}_trimed"
    print(name)
    pred_sdfs = [c for c in cases if re.search(r"rank[0-9]{1,2}_ligand_[a-z_0-9.]*_relaxed.sdf", c.split("/")[-1]) is not None]
    if len(pred_sdfs) == 0:
        continue
    pred = torch.stack([xyz_from_sdf(p) for p in pred_sdfs])
    ranks = torch.tensor([int(re.findall(r'(?<=rank)[0-9]{1,2}(?=_)', c)[0]) for c in pred_sdfs])
    ranks_sorted = ranks.argsort(descending=False)
    pred = pred[ranks_sorted]

    targets.append(target)
    preds.append(pred)
    names.append(name)

print(len(targets))
print(len(preds))
print(len(names))